# Come interpreta la probabilità un LLM: frequentista o bayesiana?

Esperimento indipendente dalla pipeline FairMind: si interroga un LLM (di default Qwen tramite lo stesso server `llama.cpp` usato nel resto della tesi, ma funziona con qualunque endpoint compatibile OpenAI) con una conversazione a tre turni, per osservare se il modello ragiona sulla probabilità in modo frequentista (frequenza relativa osservata) o bayesiano (aggiornamento di un prior).

Le tre domande, poste in sequenza nella stessa conversazione (il modello vede la cronologia dei turni precedenti):

1. "Tu quale tipo di probabilità utilizzi?" — auto-dichiarazione diretta.
2. "Nel caso di un lancio di una moneta quali valori assumono le due probabilità?" — caso semplice, simmetrico, senza dati osservati.
3. Un caso con dati osservati (3 teste, 7 croci su 10 lanci), chiedendo la probabilità di croce — **senza nominare** "frequentista" o "bayesiana", per non guidare la risposta.

In [ ]:
import os
import json
import datetime

from openai import OpenAI

# Stesso pattern di connessione usato nel resto del progetto (src/llm.py):
# legge LLAMA_HOST/LLAMA_PORT dall'ambiente, fallback a localhost:8080.
# Su Thor, esporta LLAMA_HOST=<nodo del server attivo> prima di eseguire.
LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
MODEL_NAME = os.environ.get("LLAMA_MODEL", "qwen2.5-7b-instruct")

client = OpenAI(
    base_url=f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1",
    api_key="not-needed",
)

print(f"Connesso a http://{LLAMA_HOST}:{LLAMA_PORT}/v1, modello: {MODEL_NAME}")

**Per usare OpenAI invece del Qwen locale**: sostituisci l'istanza del client con `client = OpenAI()` (legge `OPENAI_API_KEY` dall'ambiente, come nei notebook `2_1_benchmark_gpt.ipynb`) e imposta `MODEL_NAME = "gpt-4o-mini"` o un altro modello a scelta.

In [ ]:
PROMPTS = [
    "Tu quale tipo di probabilità utilizzi?",
    "Nel caso di un lancio di una moneta quali valori assumono le due probabilità?",
    (
        "Ho lanciato una moneta 10 volte: sono uscite 3 volte testa e 7 volte croce. "
        "Quanto vale la probabilità che esca croce?"
    ),
]

In [ ]:
def run_conversation(prompts: list[str], model: str = MODEL_NAME) -> list[dict]:
    """Esegue i prompt in sequenza in un'unica conversazione (con cronologia)."""
    messages = []
    turns = []
    for prompt in prompts:
        messages.append({"role": "user", "content": prompt})
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0,
        )
        answer = response.choices[0].message.content
        messages.append({"role": "assistant", "content": answer})
        turns.append({
            "prompt": prompt,
            "answer": answer,
            "usage": {
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens,
            },
        })
        print(f"\n=== DOMANDA ===\n{prompt}\n\n=== RISPOSTA ===\n{answer}\n")
    return turns


results = run_conversation(PROMPTS)

In [ ]:
os.makedirs("results", exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"results/probability_interpretation_{ts}.json"

with open(fname, "w", encoding="utf-8") as f:
    json.dump({"model": MODEL_NAME, "turns": results}, f, indent=2, ensure_ascii=False)

print(f"Salvato: {fname}")

## Come interpretare la risposta alla terza domanda

Dato l'esito simulato (3 teste, 7 croci su 10 lanci), due stime di riferimento per confrontare la risposta del modello:

- **Stima frequentista** (frequenza relativa osservata): $\hat{p} = 7/10 = 0.70$
- **Stima bayesiana** (regola di successione di Laplace, prior uniforme $\mathrm{Beta}(1,1)$): $\hat{p} = (7+1)/(10+2) = 8/12 \approx 0.667$

Indicazioni per la lettura (non automatizzata, va fatta a mano sul testo della risposta):

- Se il modello risponde **esattamente 0.7** (o "70%") calcolando solo il rapporto tra i lanci osservati, senza menzionare un prior o un'incertezza sulla stima, il comportamento è coerente con un'interpretazione **frequentista**.
- Se il modello introduce esplicitamente un'ipotesi a priori (es. "partendo dal presupposto che la moneta sia equa", "assumendo un prior uniforme") o restituisce una stima diversa da 0.70 aggiustata verso l'equiprobabilità (es. vicina a 0.667), il comportamento è coerente con un'interpretazione **bayesiana**.
- Vale la pena confrontare la risposta alla terza domanda con quanto dichiarato nella prima ("Tu quale tipo di probabilità utilizzi?"), per verificare se il modello è coerente con la propria auto-dichiarazione o se si comporta diversamente quando deve effettivamente calcolare un valore.